### CS/ECE/ISyE 524 &mdash; Introduction to Optimization &mdash; Spring 2026 ###

# GPU Task Scheduling Optimization #

#### Xuechun Jin (xjin252@wisc.edu), Jing Lan (jlan25@wisc.edu), Devon Ding (tding33@wisc.edu)

### Table of Contents

1. [Introduction](#1.-Introduction)
1. [Mathematical Model](#2.-Mathematical-model)
1. [Implementation and Solution](#3.-Implementation)
1. [Discussion of Results](#4.-Discussion-of-results)
1. [Conclusion](#5.-Conclusion)

## 1. Introduction ##

Our project aims to improve GPU task scheduling by building optimization models. It includes a basic model and an extended model, both formulated as MILP models. We aim to improve the overall system performance by efficiently assigning tasks to limited GPU resources. Under different settings, we aim to minimize the total completion time when all tasks must be executed, and maximize the total task value within a limited time when tasks can be skipped. 

In our basic model, we have 20 tasks and 4 identical GPUs. Each task has two possible configurations, each specifying the number of GPUs required and the runtime to complete the task. For example, a task can take 24 minutes when using one GPU, but might only need 14 minutes when running on two. Each task can choose only one configuration, and all tasks must be executed. We present the execution order by assigning a start time and a finish time to each task, and we ensure that the total number of GPUs in use does not exceed 4 at any time. Our goal is to choose the best configuration combination and the order of executing tasks to minimize the total completion time. 

In the extended model, each task is assigned a value to represent its priority. A higher value means a higher priority. Unlike the basic model, tasks are not required to be executed, and some tasks can be skipped as needed. In addition, there is a total execution time budget now. Our goal is to select a subset of tasks and their execution order to maximize the total value of completed tasks within the time limit. 

Mordern GPUs are widely used in large-scale computing. However, once a GPU task starts running, it usually cannot be interrupted. When multiple applications share the same GPU, important tasks may be delayed for a long time. Therefore, It is important to schedule tasks properly so that high-value tasks can be completed as early as possible. This can improve the overall system performance and make better use of expensive GPU resoueces. An early work, TimeGraph [1], propoeses a GPU scheduling method that uses task priorities to decide the execution order, with a max running time constraint to prevent one task from abusing resources. However, for simplicity, we do not consider the execution time limits in our project. Another recent work introduces the operation of a shared GPU cluster on HKUST campus [2]. GPUs are offered in packs of 1, 2, 4, and 8. While this strategy mitigates the fragmentation issues, it losses some scheduling flexibility and assumes a decent cluster.

We do not use real data for this project. Instead, we generate synthetic data for GPU requirements, runtime, and task values. We also consider the following assumptions to simplify our models:
 - All GPUs are identical and can run tasks at the same time.
 - Tasks are independent from each other.
 - Tasks cannot be interrupted once it starts execution.
 - Each task can only choose one given configuration.
 - All tasks must be executed in the basic model.
 - Tasks can be skipped in the extended model. 

In the following Methematical Model section, we build the mathematical formulations for both our basic model and the extended model, including the decision variables, objective functions, and constraints.

In the Implementation & Solutions section, we first generate the synthetic data used in this experiment and slove both models using JuMP. We also adjust the number of parallel GPUs in the basic model and the time budget in the extended model to obtain the corresponding optimal makespan and task value. The results are prepared for later discussion. In section 3.F, we modify the runtime of Task 1 to be much longer than the other tasks in order to study how the system balances task runtime and value.

In the Discussion section, based on the results from the previous section, we analyze whether the outputs of the basic and extended models are reasonable. We also discuss the effect of total GPU capacity on the optimal kamespan, the tradeoff between time limit and task value, and our observation from the long-tail task experiment. 

Finally, the Conclusion section briefly summarize the entire report and provide some possible future directions for this project. 

## 2. Mathematical model ##

## 2.A Basic Model ##

**Data:**
 - $p_{ik}$ is the running time for task $i$ with consiguration $k$
 - $g_{ik}$ is the required number of GPUs for task $i$ with consiguration $k$
 - $G$ is the total number of GPU
 - $N$ is the total number of tasks
 - $H$ is the upper bound of the total completion time: $H = \sum_{i = 1}^{N} \max_{k \in \{1, 2\}} p_{ik}$

**Decision variables:**

1. For the configuration selection and the task start time:
    $$
    x_{ikt} = \begin{cases}
    1, \quad\text{if task $i$ starts at time $t$ with configuration $k$} \\
    0, \quad\text{otherwise}
    \end{cases} \quad \forall i \in \{1, 2,..., N\}, k \in \{1, 2\}, t \in \{0, 1,..., H - p_{ik}\}
    $$

2. To evaluate the results, we define: $C$ as the total completion time (makespan) for all tasks. 

**Objective：**

Our objective is to minimize the makespan C:
    $$\min C$$

**Constraints:**

1. Each task has to choose exactly one configuration (means all tasks must be executed):
    $$\sum_{k \in \{1, 2\}}\sum_{t = 0}^{H - p_{ik}} x_{ikt} = 1 \quad \forall i \in \{1, 2,..., N\}$$
2. The total number of GPUs in use does not exceed 4 at any time point $r$.
    $$\sum_{i = 1}^{N} \sum_{k \in \{1, 2\}} \sum_{t \le r \le t + p_{ik} - 1} g_{ik}x_{ikt} \le G \quad \forall r \in \{0, 1,..., H\}$$
3. Makespan C must be greater than the completion time of every single tasks:
    $$C \ge \sum_{k \in \{1, 2\}} \sum_{t = 0}^{H - p_{ik}} (t + p_{ik})x_{ikt} \quad \forall i \in \{1, 2,..., N\}$$
4. Non-negativity:
    $$x_{ikt} \ge 0, \quad C \ge 0$$

**The full model in standard form:**
$$
\begin{aligned}
\max \quad& -C \\
\text{subject to} \quad
& \sum_{k \in \{1, 2\}}\sum_{t = 0}^{H - p_{ik}} x_{ikt} \le 1 \quad \forall i \in \{1, 2,..., N\}\\
& -\sum_{k \in \{1, 2\}}\sum_{t = 0}^{H - p_{ik}} x_{ikt} \le -1 \quad \forall i \in \{1, 2,..., N\}\\
& \sum_{i = 1}^{N} \sum_{k \in \{1, 2\}} \sum_{t \le r \le t + p_{ik} - 1} g_{ik}x_{ikt} \le G \quad \forall r \in \{0, 1,..., H\}\\
& \sum_{k \in \{1, 2\}} \sum_{t = 0}^{H - p_{ik}} (t + p_{ik})x_{ikt} \le C \quad \forall i \in \{1, 2,..., N\}\\
& x_{ikt} \ge 0, \quad C \ge 0\\
\end{aligned}
$$

## 2.B Extended Model ##

**Data:**
 - $p_{ik}$ is the running time for task $i$ with consiguration $k$
 - $g_{ik}$ is the required number of GPUs for task $i$ with consiguration $k$
 - $G$ is the total number of GPU
 - $N$ is the total number of tasks
 - $L$ is the given time limit
 - $v_i$ is the given task value for task $i$

**Decision variables:**

1. For the configuration selection and the task start time:
    $$
    x_{ikt} = \begin{cases}
    1, \quad\text{if task $i$ starts at time $t$ with configuration $k$} \\
    0, \quad\text{otherwise}
    \end{cases} \quad \forall i \in \{1, 2,..., N\}, k \in \{1, 2\}, t \in \{0, 1,..., L - p_{ik}\}
    $$


**Objective：**

Our objective is to maximize the total executed task value:
    $$\max \sum_{i = 1}^{N} \sum_{k \in \{1, 2\}}\sum_{t = 0}^{L - p_{ik}} v_ix_{ikt}$$

**Constraints:**

1. Each task can choose at most one configuration (tasks can be skipped):
    $$\sum_{k \in \{1, 2\}}\sum_{t = 0}^{L - p_{ik}} x_{ikt} \le 1 \quad \forall i \in \{1, 2,..., N\}$$
2. The total number of GPUs in use does not exceed 4 at any time point $r$:
    $$\sum_{i = 1}^{N} \sum_{k \in \{1, 2\}} \sum_{t \le r \le t + p_{ik} - 1} g_{ik}x_{ikt} \le G \quad \forall r \in \{0, 1,..., L\}$$
3. The completion time of all executed tasks cannot exceed L: Since we change the range of $t$ to $[0, L - p_{ik}]$, this requirement is met automatically.
4. Non-negativity:
    $$x_{ikt} \ge 0$$

**The full model in standard form:**
$$
\begin{aligned}
\max \quad& \sum_{i = 1}^{N} \sum_{k \in \{1, 2\}}\sum_{t = 0}^{L - p_{ik}} v_ix_{ikt} \\
\text{subject to} \quad
& \sum_{k \in \{1, 2\}}\sum_{t = 0}^{L - p_{ik}} x_{ikt} \le 1 \quad \forall i \in \{1, 2,..., N\}\\\
& \sum_{i = 1}^{N} \sum_{k \in \{1, 2\}} \sum_{t \le r \le t + p_{ik} - 1} g_{ik}x_{ikt} \le G \quad \forall r \in \{0, 1,..., L\}\\
& x_{ikt} \ge 0\\
\end{aligned}
$$

## 3. Implementation \& Solutions ##

## 3.A Model Data Generation

First, we generate synthesize data as the input for both our basic and extended model. We set a fixed random seed to ensure that we reproduce the task configurations across runs. As mentioned in the [Introduction](#1.-Introduction) section, we have 4 available GPUs and 20 tasks. For each task, we randomize its single-GPU runtime from the range of 15 to 30 minutes. We also offer a multi-device option: each task can also be processed by 2 GPUs, in return of a shorter completion time. 

In [10]:
using JuMP, HiGHS, Random

# data generated are the same everytime
Random.seed!(1218)

# the number of GPUs
G = 4
# the number of tasks
N = 20
# p is the running time. Configurations with more GPUs run faster
p = zeros(Int, N, 2)
for i in 1:N
    p[i, 1] = rand(15:30)
    p[i, 2] = rand(5:p[i, 1]-5)
end
# g: required number of GPUs for config 1 and 2
g = ones(Int, N, 2)
g[:, 2] .= 2
# H: the execution time upperbound
H = sum(maximum(p[i, :]) for i in 1:N)

println("Completion time upper bound (H): $H")
println("Running time (p):")
println(p)
println("GPU requirements (g):")
println(g)

Completion time upper bound (H): 451
Running time (p):
[24 14; 15 5; 27 6; 21 6; 23 16; 15 9; 25 9; 26 11; 18 12; 18 13; 27 9; 19 5; 26 17; 23 15; 26 11; 18 7; 25 9; 25 17; 30 23; 20 9]
GPU requirements (g):
[1 2; 1 2; 1 2; 1 2; 1 2; 1 2; 1 2; 1 2; 1 2; 1 2; 1 2; 1 2; 1 2; 1 2; 1 2; 1 2; 1 2; 1 2; 1 2; 1 2]


Next, two more types of variables are required for the extended model. From the preliminary results we obtained from the basic model (see [Basic Model Results](#4.A-Basic-Model)), the minimal makespan is 94 minutes. Therefore, we choose to pick smaller time limit L of 70 minutes as the initial parameter. Additionally, we assign a random integer to each task as the task value, reflecting the task priority. Our extended model aims to maximize the total task value within the limited execution time.

In [11]:
# time limit L
# since the min makespan is 94 mins, we need to choose a number smaller than 94
L = 70

# tasks have higher value have high priority
# v = rand(1:10, N);
v = [10, 9, 1, 1, 1, 9, 5, 2, 6, 1, 10, 5, 6, 3, 5, 5, 5, 8, 3, 1];

println("Time limit (L): $L")
println("Task values (v):")
println(v)


Time limit (L): 70
Task values (v):
[10, 9, 1, 1, 1, 9, 5, 2, 6, 1, 10, 5, 6, 3, 5, 5, 5, 8, 3, 1]


We present our model input in the following table. A total of 20 tasks are generated. The middle two columns present the runtime required for each task, using two different configurations. Config 1 uses 1 GPU and Config 2 ues 2 simultaneously. The last column shows the task values, which will be used as additional parameters of the extended model. 

| Task | Config 1 Runtime (mins) | Config 2 Runtime (mins) | Value |
|-----:|------------------------:|------------------------:|------:|
| 1    | 24                      | 14                      | 10    |
| 2    | 15                      | 5                       | 9     |
| 3    | 27                      | 6                       | 1     |
| 4    | 21                      | 6                       | 1     |
| 5    | 23                      | 16                      | 1     |
| 6    | 15                      | 9                       | 9     |
| 7    | 25                      | 9                       | 5     |
| 8    | 26                      | 11                      | 2     |
| 9    | 18                      | 12                      | 6     |
| 10   | 18                      | 13                      | 1     |
| 11   | 27                      | 9                       | 10    |
| 12   | 19                      | 5                       | 5     |
| 13   | 26                      | 17                      | 6     |
| 14   | 23                      | 15                      | 3     |
| 15   | 26                      | 11                      | 5     |
| 16   | 18                      | 7                       | 5     |
| 17   | 25                      | 9                       | 5     |
| 18   | 25                      | 17                      | 8     |
| 19   | 30                      | 23                      | 3     |
| 20   | 20                      | 9                       | 1     |

## 3.B Basic Model Implementation ##

In [12]:
m = Model(HiGHS.Optimizer)

# define x_ikt as a binary variable
@variable(m, x[i = 1:N, k = 1:2, t = 0:H - p[i,k]], Bin)  
@variable(m, C >= 0)  

@objective(m, Min, C) 
   
# each task has to choose exactly one configuration
@constraint(m, config[i in 1:N], sum(x[i, k, t] for k in 1:2, t in 0:H - p[i,k]) == 1)  
# the total number of GPUs in use does not exceed 4 at anytime
# t cannot be less than 0
@constraint(m, gpu[r in 0:H], sum(g[i, k]*x[i, k, t] 
        for i in 1:N, k in 1:2, t in max(0, r - p[i, k] + 1):min(r, H - p[i,k])) <= G)  
# makespan C must be greater than the completion time of every single tasks
@constraint(m, makespan[i in 1:N], sum((t + p[i, k])*x[i, k, t] for k in 1:2, t in 0:H - p[i,k]) <= C)  

# run optimizer
set_silent(m)
optimize!(m)


# print the results
println("The optimal result (Min makespan): ", objective_value(m), " mins.") 
for i in 1:N
    for k in 1:2
        for t in 0:H-p[i,k]
            # only print out the configs that are selected
            # we use > 0.9 instead of == 1 because sometimes the solver returns 0.999999
            if value(x[i, k, t]) > 0.9
                println("Task $i uses config $k, start time is $t, finish time = $(t + p[i, k])")
            end
        end
    end
end


The optimal result (Min makespan): 94.0 mins.
Task 1 uses config 1, start time is 19, finish time = 43
Task 2 uses config 2, start time is 5, finish time = 10
Task 3 uses config 2, start time is 9, finish time = 15
Task 4 uses config 2, start time is 24, finish time = 30
Task 5 uses config 1, start time is 19, finish time = 42
Task 6 uses config 1, start time is 61, finish time = 76
Task 7 uses config 2, start time is 37, finish time = 46
Task 8 uses config 2, start time is 83, finish time = 94
Task 9 uses config 1, start time is 76, finish time = 94
Task 10 uses config 1, start time is 43, finish time = 61
Task 11 uses config 2, start time is 10, finish time = 19
Task 12 uses config 2, start time is 0, finish time = 5
Task 13 uses config 1, start time is 46, finish time = 72
Task 14 uses config 1, start time is 46, finish time = 69
Task 15 uses config 2, start time is 72, finish time = 83
Task 16 uses config 2, start time is 30, finish time = 37
Task 17 uses config 2, start time is 15

## 3.C Extended Model Implementation ##

In [13]:
m_extended = Model(HiGHS.Optimizer)

# define x_ikt as a binary variable
@variable(m_extended, x[i = 1:N, k = 1:2, t = 0:L - p[i,k]], Bin)  

@objective(m_extended, Max, sum(v[i]*x[i, k, t] for i in 1:N, k in 1:2, t in 0:L - p[i,k]))
   
# each task has to choose at most one configuration
@constraint(m_extended, config[i in 1:N], sum(x[i, k, t] for k in 1:2, t in 0:L - p[i,k]) <= 1)  
# the total number of GPUs in use does not exceed 4 at anytime
# t cannot be less than 0
@constraint(m_extended, gpu[r in 0:L], sum(g[i, k]*x[i, k, t] 
        for i in 1:N, k in 1:2, t in max(0, r - p[i, k] + 1):min(r, L - p[i,k])) <= G)  
    
# run optimizer
set_silent(m_extended)
optimize!(m_extended)

executed = false
# print the results
println("The optimal result (Max value): ", objective_value(m_extended)) 
for i in 1:N
    executed = false
    for k in 1:2
        for t in 0:L-p[i,k]
            # only print out the executed tasks and configs
            # we use > 0.9 instead of == 1 because sometimes the solver returns 0.999999
            if value(x[i, k, t]) > 0.9
                println("Task $i uses config $k, start time is $t, finish time = $(t + p[i, k]), has value $(v[i])")
                executed = true
            end
        end
    end
    # print out skipped tasks
    if executed == false
        println("Task $i is skipped with value $(v[i])")
    end
end


The optimal result (Max value): 89.0
Task 1 uses config 1, start time is 20, finish time = 44, has value 10
Task 2 uses config 2, start time is 65, finish time = 70, has value 9
Task 3 is skipped with value 1
Task 4 uses config 2, start time is 46, finish time = 52, has value 1
Task 5 is skipped with value 1
Task 6 uses config 1, start time is 45, finish time = 60, has value 9
Task 7 uses config 2, start time is 61, finish time = 70, has value 5
Task 8 uses config 2, start time is 26, finish time = 37, has value 2
Task 9 uses config 1, start time is 20, finish time = 38, has value 6
Task 10 is skipped with value 1
Task 11 uses config 2, start time is 52, finish time = 61, has value 10
Task 12 uses config 2, start time is 60, finish time = 65, has value 5
Task 13 uses config 1, start time is 0, finish time = 26, has value 6
Task 14 uses config 1, start time is 37, finish time = 60, has value 3
Task 15 uses config 2, start time is 9, finish time = 20, has value 5
Task 16 uses config 2, s

## 3.D Effect of Total GPU Capacity on the Optimal Makespan ##

In this section, we explore the relationship between the total GPU availability and the optimal makespan from our basic model. Instead of a fixed capacity of 4 GPUs, we solve the same basic model with different GPU capacity parameters. 

In [14]:
function solveBasic(G_val)
    # the same basic model from above
    m = Model(HiGHS.Optimizer)
    @variable(m, x[i = 1:N, k = 1:2, t = 0:H - p[i,k]], Bin)  
    @variable(m, C >= 0)  
    
    @objective(m, Min, C) 
       
    # each task has to choose exactly one configuration
    @constraint(m, config[i in 1:N], sum(x[i, k, t] for k in 1:2, t in 0:H - p[i,k]) == 1)  
    # the total number of GPUs in use does not exceed G_val at any time
    # t cannot be less than 0
    @constraint(m, gpu[r in 0:H], sum(g[i, k]*x[i, k, t] 
            for i in 1:N, k in 1:2, t in max(0, r - p[i, k] + 1):min(r, H - p[i,k])) <= G_val)  
    # makespan C must be greater than the completion time of every single tasks
    @constraint(m, makespan[i in 1:N], sum((t + p[i, k])*x[i, k, t] for k in 1:2, t in 0:H - p[i,k]) <= C)  
    
    set_silent(m)
    # the model took at least 5 hours to run
    # so we set a time limit of 120s per solve 
    # and we found the difference of results is very small and can be neglected
    set_time_limit_sec(m, 120.0)
    optimize!(m)

    return objective_value(m)
end

# we consider GPU numbers from 1 to 8
G_numbers = 1:8
makespan = zeros(8)

# find the corresponding optimal makespan
for (i, G_val) in enumerate(G_numbers)
    println("Solving for total GPU capacity: $G_val...")
    makespan[i] = solveBasic(G_val)
    println("Total GPU Capacity: $G_val, Optimal Makespan: $(makespan[i]) mins.")
end

# plot the relationship between total GPU capacity and optimal makespan
using Plots
p3d = plot(G_numbers, makespan, mc=:blue, markershape=:circle, legend=false, markersize=2)
xlabel!(p3d, "Total GPU Capacity")
ylabel!(p3d, "Optimal Makespan (minutes)")
title!(p3d, "Effect of Total GPU Capacity on the Optimal Makespan")
# display(p3d)
savefig(p3d, "fig_3D.png");


Solving for total GPU capacity: 1...
Total GPU Capacity: 1, Optimal Makespan: 451.0 mins.
Solving for total GPU capacity: 2...
Total GPU Capacity: 2, Optimal Makespan: 247.0 mins.
Solving for total GPU capacity: 3...
Total GPU Capacity: 3, Optimal Makespan: 130.0 mins.
Solving for total GPU capacity: 4...
Total GPU Capacity: 4, Optimal Makespan: 94.0 mins.
Solving for total GPU capacity: 5...
Total GPU Capacity: 5, Optimal Makespan: 77.0 mins.
Solving for total GPU capacity: 6...
Total GPU Capacity: 6, Optimal Makespan: 63.0 mins.
Solving for total GPU capacity: 7...
Total GPU Capacity: 7, Optimal Makespan: 54.0 mins.
Solving for total GPU capacity: 8...
Total GPU Capacity: 8, Optimal Makespan: 47.0 mins.


## 3.E Tradeoff between the Time Limit and Task Value ##

In this experiment, the total GPU capacity remains fixed, while the time limit varies. For each value of the time limit, we solve the extended model to maximize the total task value obtained within that time budget. This allows us to examine the tradeoff between a tighter scheduling horizon and the amount of value that can be obtained.

In [15]:
function solveExtended(L_val)
    # the extended model with a variable time limit L_val
    m_extended = Model(HiGHS.Optimizer)

    # tasks with runtime longer than L_val have no feasible start time
    @variable(m_extended, x[i = 1:N, k = 1:2, t = 0:L_val - p[i,k]], Bin)  
    @objective(m_extended, Max, sum(v[i]*x[i, k, t] for i in 1:N, k in 1:2, t in 0:L_val - p[i,k]))
       
    # each task has to choose at most one configuration
    @constraint(m_extended, config[i in 1:N], sum(x[i, k, t] for k in 1:2, t in 0:L_val - p[i,k]) <= 1)  
    # the total number of GPUs in use does not exceed G at any time
    # t cannot be less than 0
    @constraint(m_extended, gpu[r in 0:L_val], sum(g[i, k]*x[i, k, t] 
            for i in 1:N, k in 1:2, t in max(0, r - p[i, k] + 1):min(r, L_val - p[i,k])) <= G)  
        
    set_silent(m_extended)
    optimize!(m_extended)
    return objective_value(m_extended)
end

# from 40 to 100, choose every 5 minutes
L_numbers = 40:5:100
values = zeros(length(L_numbers))

# find the corresponding optimal total task value
for (i, L_val) in enumerate(L_numbers)
    println("Solving for time limit: $L_val mins...")
    values[i] = solveExtended(L_val)
    println("Time Limit: $L_val mins, Optimal Total Task Value: $(values[i])")
end

# plot the tradeoff between time limit and total task value
using Plots
p3e = plot(L_numbers, values, mc=:blue, markershape=:circle, legend=false, markersize=2)
xlabel!(p3e, "Time Limit (minutes)")
ylabel!(p3e, "Optimal Total Task Value")
title!(p3e, "Tradeoff between Time Limit and Total Task Value")
# display(p3e)
savefig(p3e, "fig_3E.png");


Solving for time limit: 40 mins...
Time Limit: 40 mins, Optimal Total Task Value: 67.0
Solving for time limit: 45 mins...
Time Limit: 45 mins, Optimal Total Task Value: 72.00000000000018
Solving for time limit: 50 mins...
Time Limit: 50 mins, Optimal Total Task Value: 77.00000000000007
Solving for time limit: 55 mins...
Time Limit: 55 mins, Optimal Total Task Value: 80.99999999999994
Solving for time limit: 60 mins...
Time Limit: 60 mins, Optimal Total Task Value: 84.00000000000001
Solving for time limit: 65 mins...
Time Limit: 65 mins, Optimal Total Task Value: 86.99999999999996
Solving for time limit: 70 mins...
Time Limit: 70 mins, Optimal Total Task Value: 89.0
Solving for time limit: 75 mins...
Time Limit: 75 mins, Optimal Total Task Value: 91.0
Solving for time limit: 80 mins...
Time Limit: 80 mins, Optimal Total Task Value: 93.0
Solving for time limit: 85 mins...
Time Limit: 85 mins, Optimal Total Task Value: 94.0
Solving for time limit: 90 mins...
Time Limit: 90 mins, Optimal T

## 3.F Long-tail Task Experiment in the Extended Setup ##

In this experiment, We want to test how the extended model responds to a long-tail task with a much longer runtime. We fix the time limit at 70 minutes and modify Task 1 so that it takes 60 minutes with Config 1 and 30 minutes with Config 2. We then test task 1's value from 4 to 6 to find the threshold that the optimizer switches from skipping it to executing it.

In [18]:
using JuMP, HiGHS, Random

function solveExtended(p_data, v_data, L_val)
    # the extended model with a new variable p_data and v_data to replace original variables
    m_extended = Model(HiGHS.Optimizer)

    @variable(m_extended, x[i = 1:N, k = 1:2, t = 0:L_val - p_data[i,k]], Bin)
    @objective(m_extended, Max, sum(v_data[i] * x[i, k, t] for i in 1:N, k in 1:2, t in 0:L_val - p_data[i,k]))

    @constraint(m_extended, config[i in 1:N], sum(x[i, k, t] for k in 1:2, t in 0:L_val - p_data[i,k]) <= 1)
    @constraint(m_extended, gpu[r in 0:L_val], sum(g[i, k] * x[i, k, t]
            for i in 1:N, k in 1:2, t in max(0, r - p_data[i, k] + 1):min(r, L_val - p_data[i,k])) <= G)

    set_silent(m_extended)
    set_time_limit_sec(m_extended, 120.0)
    optimize!(m_extended)

    # initialization, assume none of the tasks are selected
    selected = falses(N)
    chosen_config = zeros(Int, N)
    start_time = fill(-1, N)

    # record tasks that are executed
    for i in 1:N, k in 1:2, t in 0:L_val - p_data[i,k]
        if value(x[i, k, t]) > 0.9
            selected[i] = true
            chosen_config[i] = k
            start_time[i] = t
        end
    end

    return termination_status(m_extended), objective_value(m_extended), selected, chosen_config, start_time
end

# change the config 1 and 2 of task 1 to 60 and 30
L_fixed = 70
p_longtail = copy(p)
p_longtail[1, 1] = 60
p_longtail[1, 2] = 30

# test value from 4 to 6
for v1 in 4:6
    # only change the task value of task 1
    v_longtail = copy(v)
    v_longtail[1] = v1

    status, obj, selected, chosen_config, start_time = solveExtended(p_longtail, v_longtail, L_fixed)

    # print out which tasks are executes and observe the changes
    println("Task 1 value = $v1, total value = $obj")

    # only printout the status of task 1, suppress others 
    if selected[1]
        println("   Task 1: SELECTED (Config $(chosen_config[1]), start = $(start_time[1]) min)")
    else
        println("   Task 1: SKIPPED")
    end
    
end


Task 1 value = 4, total value = 81.0
   Task 1: SKIPPED
Task 1 value = 5, total value = 81.0
   Task 1: SKIPPED
Task 1 value = 6, total value = 82.0
   Task 1: SELECTED (Config 1, start = 0 min)


## 4. Discussion of Results ##

## 4.A Basic Model ##

**Solver Result：**

The optimal makespan for the generated task set is 94 minutes. Note that the basic model doesn't require two consecutive GPUs for the second configuration. For example, Task 7 uses 2 GPUs but they do not need to be physically adjacent. This setup provides scheduling flexibility. We find that there are multiple feasible schedule sharing this makespan, and the following two Gantt figures show two possible schedules. 

<img src="fig_basic_gantt.png">

<center>Figure 1: Feasible Schedule 1 for the Basic Model (generated with LLM assistance based on our solver's output)</center>

<img src="fig_basic_gantt_1.png">

<center>Figure 2: Feasible Schedule 2 for the Basic Model (generated with LLM assistance based on our solver's output)</center>

**Effect of Total GPU Capacity on the Optimal Makespan:**

The following figure is the output of 3.D section and reveals the relationship between the optimal makespan and the GPU capacity $G$. We can see that optimal makespan when $G = 4$ is consistent with the result we obtained from section 3.B. Since any task can run with only 1 GPU (config 1), the optimal makespan equals to the upperbonund execution time $H$ when only 1 device is offered. We observe that the optimal makespan monotonously decreases as the number of available GPUs increases. We think this is because with more GPUs available, tasks using Config 2 can be scheduled more freely and allowing greater parallelism.  

<img src="fig_3D.png">

<center>Figure 3: Effect of Total GPU Capacity on the Optimal Makespan</center>

## 4.B Extended Model ##

**Solver Result:**

Given the 70-minute time limit, our extended model reports a maximum task value of 89, which is very close to the total value of the generated task set, 96. From the output, we can see that the skipped tasks are Task 3, 5, 10, 19, and 20, with values 1, 1, 1, 3, and 1 respectively. Meanwhile, most of the executed tasks have higher values except Task 4, 8, and 14. In addition, skipped tasks also have relatively higher runtime, so we think this result is reasonable. We also tested this model a few times, unlike the basic model, the task selection seems to be unique across multiple runs.

We save 25.5% of the execution time (from 94 mins to 70 mins) at the cost of just 7.3% ((96-89)/96 = 7.3%) task value, which shows that our model tends to skip tasks with a lower value/runtime ratio. 

The following figure illustrates an output schedule.

<img src="fig_extended_gantt.png">

<center>Figure 4: Schedule for Extended Model (generated with LLM assistance based on our solver's output)</center>

**Tradeoff between the Time Limit and Task Value**

We further explore the tradeoff between the schedule time limit and the optimal achievable total task value. The figure below visualizes the relationship. The result at $L=70$ is consistent with the result we obtained in 3.C section. 

From Figure 5 we can see that the optimal total task value increases as the time limit grows. The curve reaches its maximum value of 96 arond L = 94 minutes, which is the maximum makespan needed for all tasks to be executed. Beyond that point, all tasks can be completed within the time limit, so the total value remains the same. This shows that our model behaves as expected. 


<img src="fig_3E.png">

<center>Figure 5: Tradeoff between the Time Limit and Task Value</center>

**Long-tail Task Experiment:**

In this experiment, we modify Task 1 into a long-tail task, which means its runtime is significantly longer than the other tasks. We change its Config 1 runtime to 60 minutes and Config 2 runtime to 30 minutes, while keeping all other tasks unchanged. We then vary Task 1's value from 4 to 6 and solve the extended model for each case to study how the optimizer adjusts its task selection under a constrained time budget (L = 70).

We observe a turning point between value 5 and value 6. When Task 1's value is less or equal to 5, the optimizer skips it in favor of shorter, higher-efficiency tasks, as shown in the Gantt chart below.

<img src="fig_longtail_v5.png">

<center>Figure 6: Task 1 Skipped with Value 5 (generated with LLM assistance based on our solver's output)</center>

Once the value reaches 6, however, the optimizer selects Task 1 using Config 1 and drops Tasks 8 and 19 to free up the necessary GPU time, increasing the total value from 81 to 82.

<img src="fig_longtail_v6.png">

<center>Figure 7: Task 1 Selected with Value 6 (generated with LLM assistance based on our solver's output)</center>

We believe the task selection might be related to tasks' value runtime ratio. For example, both configurations of the modified Task 1 needs exactly 60 minutes to execute ($60 \times 1 = 30 \times 2 = 60$), so its ratio at value 5 is $\frac{5}{60} = 0.083$. The two displaced tasks are Task 8 and Task 19. Task 8 has value 2 and chooses Config 2, so it needs $11 \times 2 = 22$ minutes to execute. Task 19 has value 3 and chooses Config 1, so it needs $30 \times 1 = 30$ minutes to execute. Together they need $22 + 30 = 52$ minutes for a combined value of $2 + 3 = 5$. Their ratio is $\frac{5}{52}=0.096$, which is greater than $0.083$, so the solver tends to choose Task 8 and Task 19. 

However, when we set the Task 1's value to 6, the ratio becomes $\frac{6}{60} = 0.1$, which is greater than $0.096$, so the solver replaces Task 8 and Task 19 with Task 1.

**Model Limitations:**

In our model, in order to simplify the mathematical formulation, we discretize time into 1 minute intervals instead of treating it as continuous as in real life. This may reduce the accuracy of the results, and makes the model less effective when solving real-life problems, since in reality tasks can start at any time rather than only at integer minutes. 

In addition, for models in Section 3.D and 3.F, we found that the computation time increases significantly. Each run can take several hours without any limitations. Therefore, to make our project managable, we added set_time_limit_sec to limit each solve to 120 seconds. This inevitably sacrifice some result accuracy. If time allows, increasing the time limit or run models fully may further improve the performance of our models. 

## 5. Conclusion ##

**Summarize findings and results:**

Through the experiments of the basic model, we found that even the optimal makespan remains the same, the system can schedule tasks in different ways and there is no unique solution. As the number of available GPUs increases, the level of parallelism increases, making task scheduling more flexible. Therefore, as a result, the minimum makespan needed to execute every tasks decreases, and the overall system efficiency improves. 

From the experiments of the extended model, we found that the system tends to skip tasks with longer runtime and has lower task values in order to achieve a higher total task value within a time limit. Combined with the result from the Long-tail Task Experiment, where the status of Task 1 changed from being skipped to being selected when its value increases from 5 to 6, we observed that a task's value runtime ratio is important in determining whether it will be selected by the system. Finally, by expanding the time limit range from 70 to [40, 100], we found that the optimal task value increases as the time limit becomes longer. 

**Future directions:**

Currently, due to time limitation, each task in our model only has two configurations. In the future, the configuration choices could be expended. For example, we can add options that allow a task to run with 4 or 8 GPUs simultaneously, which would give the system higher flexibility and the situation is closer to real-world scenarios. 

In addition, the tasks in our model are currently independent from each other. I believe that adding task dependencies would be a good future direction for future study. For example, Task 3 may depend on Task 10 and Task 17, which means it can only start executing after these two tasks are completed. One possible approach I could think of is to divide tasks into layers. Tasks within the same layer are independent, but tasks in earlier layers are reuqired be completed before tasks in later layers can start. This would make the model more realistic and more likely to solve real-world problems, but it would also make the model much more complex.

## 6. References ##

1. Kato, Shinpei, et al. "{TimeGraph}:{GPU} Scheduling for {Real-Time}{Multi-Tasking} Environments." 2011 USENIX Annual Technical Conference (USENIX ATC 11). 2011.
2. Xu, Kaiqiang, et al. "Design and operation of shared machine learning clusters on campus." Proceedings of the 30th ACM International Conference on Architectural Support for Programming Languages and Operating Systems, Volume 1. 2025.
3. Claude (Sonnet 4.6), Anthropic. Gantt chart visualizations generated based on solver output. 2026.

## 6. Appendix ##

LLM-Assisted Visualization: The Gantt charts in this report (Figure 1, 2, 4, 6, and 7) were generated with the assistance of Claude (Sonnet 4.6) based on the output of our optimization solver. 